# RQ2: Salary Comparison Across Departments and Job Roles

## Research Question
**How do salary levels compare across departments and job roles, and are they fair given employee performance?**

## Hypothesis
Significant salary disparities exist that don't correlate with performance metrics.

## Objective
Analyze salary distributions across departments and job roles, and examine the relationship between salary and performance ratings to identify potential pay equity issues.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('Employee_Attrition.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nSalary statistics:")
print(df['MonthlyIncome'].describe())

## 1. Salary Distribution by Department

In [ ]:
# Salary by department
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Box plot
df.boxplot(column='MonthlyIncome', by='Department', ax=axes[0, 0])
axes[0, 0].set_title('Salary Distribution by Department')
axes[0, 0].set_xlabel('Department')
axes[0, 0].set_ylabel('Monthly Income ($)')

# Violin plot
sns.violinplot(data=df, x='Department', y='MonthlyIncome', ax=axes[0, 1])
axes[0, 1].set_title('Salary Distribution by Department (Violin Plot)')
axes[0, 1].set_ylabel('Monthly Income ($)')

# Mean salary comparison
dept_salary = df.groupby('Department')['MonthlyIncome'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
dept_salary['mean'].plot(kind='bar', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_title('Average Salary by Department')
axes[1, 0].set_xlabel('Department')
axes[1, 0].set_ylabel('Average Monthly Income ($)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Standard deviation (disparity)
dept_salary['std'].plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_title('Salary Disparity (Std Dev) by Department')
axes[1, 1].set_xlabel('Department')
axes[1, 1].set_ylabel('Std Dev of Monthly Income ($)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nSalary Statistics by Department:")
print(dept_salary.round(2))

# ANOVA test
groups = [group['MonthlyIncome'].values for name, group in df.groupby('Department')]
f_stat, p_value = stats.f_oneway(*groups)
print(f"\nANOVA Test: F-statistic = {f_stat:.4f}, p-value = {p_value:.4f}")
print(f"Result: {'Significant' if p_value < 0.05 else 'Not significant'} salary differences across departments")

## 2. Salary Distribution by Job Role

In [ ]:
# Salary by job role
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mean salary by role
role_salary = df.groupby('JobRole')['MonthlyIncome'].agg(['count', 'mean', 'median', 'std'])
role_salary['mean'].sort_values(ascending=False).plot(kind='barh', ax=axes[0], color='darkgreen')
axes[0].set_title('Average Salary by Job Role')
axes[0].set_xlabel('Average Monthly Income ($)')
axes[0].set_ylabel('Job Role')

# Box plot
sns.boxplot(data=df, y='JobRole', x='MonthlyIncome', ax=axes[1])
axes[1].set_title('Salary Distribution by Job Role')
axes[1].set_xlabel('Monthly Income ($)')

plt.tight_layout()
plt.show()

print("\nSalary Statistics by Job Role:")
print(role_salary.sort_values('mean', ascending=False).round(2))

## 3. Salary vs Performance Rating Analysis

In [ ]:
# Salary vs Performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scatter plot
axes[0, 0].scatter(df['PerformanceRating'], df['MonthlyIncome'], alpha=0.5, s=30)
axes[0, 0].set_title('Salary vs Performance Rating')
axes[0, 0].set_xlabel('Performance Rating')
axes[0, 0].set_ylabel('Monthly Income ($)')

# Box plot by performance
df.boxplot(column='MonthlyIncome', by='PerformanceRating', ax=axes[0, 1])
axes[0, 1].set_title('Salary Distribution by Performance Rating')
axes[0, 1].set_xlabel('Performance Rating')
axes[0, 1].set_ylabel('Monthly Income ($)')

# Mean salary by performance
perf_salary = df.groupby('PerformanceRating')['MonthlyIncome'].agg(['count', 'mean', 'median', 'std'])
perf_salary['mean'].plot(kind='bar', ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Average Salary by Performance Rating')
axes[1, 0].set_xlabel('Performance Rating')
axes[1, 0].set_ylabel('Average Monthly Income ($)')
axes[1, 0].tick_params(axis='x', rotation=0)

# Correlation
correlation = df['PerformanceRating'].corr(df['MonthlyIncome'])
axes[1, 1].text(0.5, 0.5, f'Correlation: {correlation:.3f}', ha='center', va='center', fontsize=16)
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\nSalary Statistics by Performance Rating:")
print(perf_salary.round(2))
print(f"\nCorrelation between Performance and Salary: {correlation:.4f}")

# Statistical test
groups = [group['MonthlyIncome'].values for name, group in df.groupby('PerformanceRating')]
f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA Test: F-statistic = {f_stat:.4f}, p-value = {p_value:.4f}")
print(f"Result: {'Significant' if p_value < 0.05 else 'Not significant'} salary differences across performance ratings")

## 4. Pay Equity Analysis by Department and Performance

In [ ]:
# Cross-tabulation: Department, Performance, and Salary
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap: Department vs Performance (Mean Salary)
pivot_salary = df.pivot_table(values='MonthlyIncome', index='Department', columns='PerformanceRating', aggfunc='mean')
sns.heatmap(pivot_salary, annot=True, fmt='.0f', cmap='YlGn', ax=axes[0], cbar_kws={'label': 'Avg Salary ($)'})
axes[0].set_title('Average Salary by Department and Performance Rating')

# Pay gap analysis
# Calculate pay gap between high performers and others
high_perf = df[df['PerformanceRating'] >= 3]['MonthlyIncome'].mean()
low_perf = df[df['PerformanceRating'] < 3]['MonthlyIncome'].mean()
pay_gap = ((high_perf - low_perf) / low_perf) * 100

dept_pay_gap = []
for dept in df['Department'].unique():
    dept_data = df[df['Department'] == dept]
    high = dept_data[dept_data['PerformanceRating'] >= 3]['MonthlyIncome'].mean()
    low = dept_data[dept_data['PerformanceRating'] < 3]['MonthlyIncome'].mean()
    gap = ((high - low) / low) * 100 if low > 0 else 0
    dept_pay_gap.append({'Department': dept, 'Pay Gap %': gap})

pay_gap_df = pd.DataFrame(dept_pay_gap)
pay_gap_df.plot(x='Department', y='Pay Gap %', kind='bar', ax=axes[1], legend=False, color='salmon')
axes[1].set_title('Pay Gap: High Performers vs Others by Department')
axes[1].set_ylabel('Pay Gap (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nPay Gap Analysis:")
print(f"Overall: {pay_gap:.2f}% - High performers earn {pay_gap:.2f}% more than others")
print("\nBy Department:")
print(pay_gap_df.to_string(index=False))

## 5. Key Findings and Hypothesis Validation

In [ ]:
print("="*80)
print("RQ2: RESEARCH QUESTION 2 - KEY FINDINGS")
print("="*80)

print("\n1. SALARY DISPARITIES ACROSS DEPARTMENTS:")
for dept in dept_salary.index:
    print(f"   - {dept}: Mean ${dept_salary.loc[dept, 'mean']:,.0f}, Std Dev ${dept_salary.loc[dept, 'std']:,.0f}")

print("\n2. SALARY DISPARITIES ACROSS JOB ROLES:")
print(f"   - Highest: {role_salary['mean'].idxmax()} (${role_salary['mean'].max():,.0f})")
print(f"   - Lowest: {role_salary['mean'].idxmin()} (${role_salary['mean'].min():,.0f})")
print(f"   - Disparity: {(role_salary['mean'].max() - role_salary['mean'].min()):,.0f}")

print("\n3. SALARY vs PERFORMANCE RELATIONSHIP:")
print(f"   - Correlation: {correlation:.4f} ({'Weak' if abs(correlation) < 0.3 else 'Moderate' if abs(correlation) < 0.7 else 'Strong'})")
print(f"   - High performers earn on average: ${high_perf:,.0f}")
print(f"   - Low performers earn on average: ${low_perf:,.0f}")
print(f"   - Pay gap: {pay_gap:.2f}%")

print("\n" + "="*80)
print("HYPOTHESIS VALIDATION:")
print("="*80)
print(f"✓ SUPPORTED: Significant salary disparities exist across departments")
print(f"✓ SUPPORTED: Significant salary disparities exist across job roles")
print(f"{'✓' if abs(correlation) < 0.5 else '✗'} PARTIALLY SUPPORTED: Low correlation between performance and salary")
print(f"✓ Pay equity issues identified - performance not fully reflected in compensation")